# CNN Model Training for Diabetic Ulcer Detection

This notebook implements and trains a CNN model for detecting diabetic ulcers in medical images.

## 1. Environment Setup and Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import logging
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Configure device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Load and Explore Clinical Data

In [ ]:
# Load clinical data
clinical_data = pd.read_csv('../datasets/clinical_data/patient_data.csv')

print(f'Dataset shape: {clinical_data.shape}')
print(f'\nFirst few rows:')
print(clinical_data.head())
print(f'\nData types:')
print(clinical_data.dtypes)
print(f'\nMissing values:')
print(clinical_data.isnull().sum())
print(f'\nBasic statistics:')
print(clinical_data.describe())

## 3. Image Data Loading and Preprocessing

In [ ]:
class UlcerDataset(Dataset):
    """Custom dataset for ulcer images."""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('Transforms configured successfully')

## 4. CNN Model Architecture and Training

In [ ]:
class CNNUlcerModel(nn.Module):
    """CNN model for ulcer classification."""
    
    def __init__(self, num_classes=3, pretrained=True):
        super(CNNUlcerModel, self).__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        
        # Replace final layer
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# Create model
model = CNNUlcerModel(num_classes=3, pretrained=True).to(DEVICE)
print(f'Model created: {model}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    """Validate model performance."""
    model.eval()
    total_loss = 0
    predictions = []
    targets = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            predictions.extend(outputs.argmax(1).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(targets, predictions)
    return total_loss / len(dataloader), accuracy

print('Training functions defined')

## 5. Model Training and Evaluation

In [ ]:
# Training configuration
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Note: In practice, you would load actual image datasets here
# For now, we'll create dummy data
print('Training configuration complete')
print(f'Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}')

## 6. Model Evaluation and Metrics

In [ ]:
def compute_metrics(y_true, y_pred, num_classes=3):
    """Compute classification metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm
    }

print('Metrics function defined')

## 7. Explainability with Grad-CAM

In [ ]:
class GradCAM:
    """Grad-CAM implementation for visualization."""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        target = self.model.backbone.layer4
        target.register_forward_hook(forward_hook)
        target.register_backward_hook(backward_hook)
    
    def generate_cam(self, input_tensor, target_class):
        """Generate CAM heatmap."""
        output = self.model(input_tensor)
        self.model.zero_grad()
        
        target_score = output[0, target_class]
        target_score.backward()
        
        weights = torch.mean(self.gradients[0], dim=(1, 2))
        cam = torch.zeros(self.activations.shape[1:], device=self.activations.device)
        
        for i, w in enumerate(weights):
            cam += w * self.activations[0, i, :, :]
        
        cam = torch.relu(cam)
        cam = cam / (cam.max() + 1e-8)
        
        return cam.cpu().numpy()

print('Grad-CAM implemented')

## 8. Generate Clinical Reports

In [ ]:
def generate_clinical_report(prediction, confidence, clinical_data):
    """Generate a clinical report."""
    classes = ['Normal', 'Ulcer', 'Severe Ulcer']
    risk_levels = ['Low', 'Moderate', 'High']
    
    report = {
        'prediction_class': classes[prediction],
        'confidence': float(confidence),
        'risk_level': risk_levels[prediction],
        'clinical_data': clinical_data,
        'recommendations': []
    }
    
    # Add recommendations based on prediction
    if prediction == 0:
        report['recommendations'] = ['Continue regular foot care', 'Schedule annual check-up']
    elif prediction == 1:
        report['recommendations'] = ['Schedule immediate medical appointment', 'Increase monitoring frequency']
    else:
        report['recommendations'] = ['Seek urgent medical attention', 'Consider specialist referral']
    
    return report

print('Report generation function defined')